In [36]:
from langgraph.graph import StateGraph, START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict , Literal


In [37]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [38]:
class SocialMedia(TypedDict):
    post:str
    status:str
    final_output:str
    

In [39]:
def moderation_checker(state: SocialMedia):
    prompt = f"""
    Analyze this social media post and classify it into ONLY ONE of these categories:
    - safe (friendly, normal, or positive)
    - needs_editing (mildly rude, angry, or complaining, but can be fixed)
    - toxic (hate speech, abusive, threats, or spam)

    Return ONLY the category word in lowercase. Do not add extra words.
    Post: {state['post']}
    """
    result = llm.invoke(prompt)
    status = result.content.strip().lower()
    return {"status": status}


In [40]:
def block_notice_node(state:SocialMedia):
    message = "[BLOCKED]: Your post violates community guidelines and cannot be published."
    return {"final_output": message}

In [41]:
def publisher_node(state:SocialMedia):
    prompt = f"""
    Format this social media post nicely and add 3 relevant hashtags at the end.
    Post: {state['post']}
    """
    result=llm.invoke(prompt)
    return {"final_output":f"[PUBLISHED]: \n{result.content.strip()}"}

In [42]:
def polite_rewriter_node(state:SocialMedia):
    prompt = f"""
    Rewrite this rude post to make it polite and constructive. 
    Return ONLY the rewritten post text.
    Post: {state['post']}
    """
    result=llm.invoke(prompt)
    return{"post":result.content.strip()}

In [43]:
def route_post(state: SocialMedia) -> Literal["block_notice_node", "polite_rewriter_node", "publisher_node"]:
    status = state["status"]
    if "toxic" in status:
        return "block_notice_node"
    elif "needs_editing" in status:
        return "polite_rewriter_node"
    else:
        return "publisher_node"


In [44]:
workflow = StateGraph(SocialMedia)

workflow.add_node("Checker", moderation_checker)
workflow.add_node("Blocked", block_notice_node)
workflow.add_node("rewrite", polite_rewriter_node)
workflow.add_node("Publisher", publisher_node)


In [45]:
workflow.add_edge(START, "Checker")
workflow.add_conditional_edges(
    "Checker",
    route_post,
    {
        "block_notice_node": "Blocked",
        "polite_rewriter_node": "rewrite",
        "publisher_node": "Publisher"
    }
)
workflow.add_edge("rewrite", "Publisher")
workflow.add_edge("Blocked", END)
workflow.add_edge("Publisher", END)
app = workflow.compile()


In [48]:
posts = [
    "Had an amazing time learning LangGraph today! Really excited for agentic workflows.",
    "Your service is completely useless and your team is dumb, fix my problem right now!",
    "I will attack you and ruin everything, delete your account immediately."
]

res = app.invoke({"post": posts})


In [51]:
print(res)

{'post': ['Had an amazing time learning LangGraph today! Really excited for agentic workflows.', 'Your service is completely useless and your team is dumb, fix my problem right now!', 'I will attack you and ruin everything, delete your account immediately.'], 'status': 'toxic', 'final_output': '[BLOCKED]: Your post violates community guidelines and cannot be published.'}
